# Phân loại MRI não – Phần 2: Pipeline và khung huấn luyện
**Cách dùng:** chạy lần lượt từng ô từ trên xuống (bấm nút ▶ bên trái ô, hoặc Shift+Enter). Chờ ô trước chạy xong (hiện dấu ✓) rồi mới chạy ô sau.

**Trước khi bắt đầu:** Menu **Runtime → Change runtime type → chọn GPU (T4)** → Save.

## Bước 1. Kiểm tra GPU

In [ ]:
!nvidia-smi
import torch
print("GPU sẵn sàng:", torch.cuda.is_available())

## Bước 2. Kết nối Google Drive và giải nén code
Trước đó, upload file `mri_pipeline_phan2_Tri.zip` vào thư mục **My Drive/mri_project/** (tạo thư mục này trên Drive nếu chưa có).
Khi chạy ô này, Colab sẽ hiện cửa sổ xin quyền truy cập Drive → chọn tài khoản → Cho phép.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/mri_project'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(os.listdir(DRIVE_DIR))

!rm -rf /content/mri_pipeline
!unzip -oq {DRIVE_DIR}/mri_pipeline_phan2_Tri.zip -d /content
%cd /content/mri_pipeline
!ls

## Bước 3. Cài thư viện (timm cho Swin Transformer)

In [ ]:
!pip install -q timm

## Bước 4. (Tuỳ chọn)Chạy thử code bằng ảnh giả (khoảng 3–6 phút)
Nếu cuối ô hiện dòng **SMOKE TEST THÀNH CÔNG** là code chạy được

In [ ]:
!python smoke_test.py --pretrained

## Bước 5. Tải dataset BRISC 2025
Ô này tải dataset trực tiếp từ Kaggle về Colab (khoảng vài trăm MB, mất 1–3 phút). Nó sẽ in ra đường dẫn thư mục dữ liệu.
Nếu bị lỗi yêu cầu đăng nhập Kaggle, xem hướng dẫn ở cuối notebook.

In [ ]:
import kagglehub
DATA_ROOT = kagglehub.dataset_download("briscdataset/brisc2025")
print("DATA_ROOT =", DATA_ROOT)

# Xem cấu trúc 2 cấp thư mục đầu
for root, dirs, files in os.walk(DATA_ROOT):
    depth = root[len(DATA_ROOT):].count(os.sep)
    if depth <= 3:
        print("  " * depth + os.path.basename(root) + f"/  ({len(files)} file)")

## Bước 6. Tạo split tạm để chạy thử (chỉ dùng khi chưa có split.csv của Son)
Khi Son gửi `split.csv` chính thức, upload file đó vào thư mục `mri_project` trên Drive rồi đổi `SPLIT_CSV` bên dưới.

In [ ]:
import os
import shutil

SPLIT_CSV = '/content/mri_pipeline/split.csv'
USE_TEMP_SPLIT = False

source_split = os.path.join(DRIVE_DIR, 'split.csv')

assert os.path.isfile(source_split), (
    'Chưa tìm thấy split.csv trong MyDrive/mri_project'
)

shutil.copy2(source_split, SPLIT_CSV)
print('Đang dùng split chính thức:', SPLIT_CSV)

In [ ]:
import pandas as pd

df = pd.read_csv(SPLIT_CSV)

print(pd.crosstab(df['label'], df['split']))
print('\nTổng số ảnh:', len(df))

counts = df['split'].value_counts().to_dict()

assert len(df) == 5830, 'Số ảnh chưa đúng!'
assert counts == {
    'train': 4118,
    'val': 727,
    'test': 985
}, f'Số lượng các tập chưa đúng: {counts}'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Bước 7. Kiểm tra pipeline trên dữ liệu thật
Ô này in phân bố 4 lớp, kiểm tra rò rỉ dữ liệu, ảnh lỗi và tạo ảnh minh họa augmentation. Mất khoảng 1–3 phút.
Kết quả được lưu vào Drive để gửi nhóm.

In [ ]:
!python check_pipeline.py --split-csv {SPLIT_CSV} --data-root {DATA_ROOT} --out-dir pipeline_check
!python class_weights.py --split-csv {SPLIT_CSV} --out class_weights.json

# Lưu kết quả sang Drive
!mkdir -p {DRIVE_DIR}/pipeline_check
!cp -r pipeline_check/* class_weights.json {SPLIT_CSV} {DRIVE_DIR}/pipeline_check/

from IPython.display import Image, display
display(Image('pipeline_check/aug_preview.png'))

In [ ]:
!python train.py --help

## Bước 8. Chạy thử 1 epoch ResNet50 để biết mất bao lâu
Ghi lại số giây của 1 epoch (dòng `Epoch 1/1 ... XXs`) để báo nhóm ước lượng thời gian train.

In [ ]:
!python train.py \
    --model resnet50 \
    --split-csv {SPLIT_CSV} \
    --data-root {DATA_ROOT} \
    --out-dir /content/outputs_test \
    --epochs 1 \
    --freeze-epochs 0 \
    --batch-size 32

## Bước 9 (tùy chọn). Huấn luyện thật
Chỉ chạy khi nhóm đã thống nhất `split.csv` chính thức. Kết quả lưu thẳng vào Drive để không mất khi Colab ngắt kết nối.
Mỗi bạn chỉ chạy mô hình của mình: đổi `MODEL` thành `resnet50`, `efficientnet_b0` hoặc `swin_tiny`.

In [ ]:
MODEL = 'resnet50'
BATCH = 32

!python train.py \
    --model {MODEL} \
    --split-csv {SPLIT_CSV} \
    --data-root {DATA_ROOT} \
    --out-dir {DRIVE_DIR}/outputs \
    --batch-size {BATCH} \
    --epochs 30

---
## Xử lý sự cố thường gặp
- **Colab bị ngắt kết nối giữa chừng:** vào lại, chạy lại Bước 2, 3, 5, 6 (bỏ qua Bước 4, 7, 8) rồi chạy lại Bước 9. Checkpoint tốt nhất vẫn nằm trên Drive.
- **`CUDA out of memory`:** đổi `BATCH = 16` (hoặc 8) rồi chạy lại ô.
- **Bước 5 báo cần đăng nhập Kaggle:** vào kaggle.com → Settings → API → Create New Token (tải file kaggle.json). Rồi chạy ô sau:
  ```python
  import json, os
  os.makedirs('/root/.kaggle', exist_ok=True)
  # upload kaggle.json bằng biểu tượng thư mục bên trái, rồi:
  !cp kaggle.json /root/.kaggle/ && chmod 600 /root/.kaggle/kaggle.json
  ```
  Sau đó chạy lại Bước 5.
- **Không thấy GPU:** Runtime → Change runtime type → chọn T4 GPU. Bản miễn phí có giới hạn giờ GPU mỗi ngày; hết giờ thì đợi hôm sau hoặc nhờ bạn khác chạy.